In [23]:
import pandas as pd
import os
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

In [24]:
pd.set_option('display.max_columns', None)

#### <span style="color:#FF6347;">**READ**</span> mo_cancer_dicom

In [25]:
file_path = os.path.join("J:\Testing\jlee\MO-DBT-Eligible\Metadata", 'mo_cancer_dicom' + ".xlsx")
df = pd.read_excel(file_path)

## <span style="color:cyan">**Function**</span> 

In [26]:
# ── Validity check: must have DBT AND (C VIEW or IN2D) ───────────────────────
def is_valid(series_set):
    has_dbt    = 'DBT' in series_set
    has_2d     = bool(series_set & {'C VIEW', 'IN2D'})
    return has_dbt and has_2d

In [27]:
# ── Distribution: DBT / DBT_Slab / Both ──────────────────────────────────────
def categorize(row):
    has_dbt      = 'DBT' in row['series_set']
    has_dbt_slab = row['has_slab']
    if has_dbt and has_dbt_slab: return 'Both DBT and DBT_Slab'
    if has_dbt:                  return 'DBT'
    if has_dbt_slab:             return 'DBT_Slab'
    return 'Neither'

## **INDEX-1**

### **Check 1**: PID with both DBT + (C VIEW / IN2D)

In [28]:
# ── Per-patient series sets at INDEX-1 ───────────────────────────────────────
idx = df[df['Index'] == 'INDEX-1'].copy()
idx['EXAM_COMPLETED_DATE'] = pd.to_datetime(idx['EXAM_COMPLETED_DATE'], errors='coerce')
patient_series = idx.groupby('PATIENT_STUDY_ID')['Series'].apply(set)
patient_has_slab = idx[idx['Slab'] == 'Y'].groupby('PATIENT_STUDY_ID').size() > 0

valid_pids   = patient_series[patient_series.apply(is_valid)].index
invalid_pids = patient_series[~patient_series.apply(is_valid)].index

print(f"INDEX-1 patients with both DBT + (C VIEW/IN2D) : {len(valid_pids)}")
print(f"INDEX-1 patients MISSING C VIEW/IN2D           : {len(invalid_pids)}")

INDEX-1 patients with both DBT + (C VIEW/IN2D) : 196
INDEX-1 patients MISSING C VIEW/IN2D           : 68


In [29]:
df_invalid = idx[idx['PATIENT_STUDY_ID'].isin(invalid_pids)]
df_invalid

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,PatientBirthDate,StudyDate,Study,SIDE,Series,View,Slab,StudyDescription,SeriesDescription,SeriesNumber,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,time_to_INDEX_months,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,Ki-67_Percent,Ki-67_Grade,subtype,FolderPath,Density,TargetFolderPath
36,4330434822,40,77664732,1978-07-01,2018-08-16,SCREEN,L,DBT,CC,NaN,Standard Screening - TomoHD,L CC Breast Tomosynthesis Image,73200000,Extremely dense (>75%),2 - Benign finding,N-Normal interval follow-up,2018-08-16,NaN,NaN,NaN,INDEX-1,15.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,D,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330434...
37,4330434822,40,77664732,1978-07-01,2018-08-16,SCREEN,L,DBT,MLO,NaN,Standard Screening - TomoHD,L MLO Breast Tomosynthesis Image,73200000,Extremely dense (>75%),2 - Benign finding,N-Normal interval follow-up,2018-08-16,NaN,NaN,NaN,INDEX-1,15.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,D,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330434...
38,4330434822,40,77664732,1978-07-01,2018-08-16,SCREEN,R,DBT,CC,NaN,Standard Screening - TomoHD,R CC Breast Tomosynthesis Image,73200000,Extremely dense (>75%),2 - Benign finding,N-Normal interval follow-up,2018-08-16,NaN,NaN,NaN,INDEX-1,15.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,D,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330434...
39,4330434822,40,77664732,1978-07-01,2018-08-16,SCREEN,R,DBT,MLO,NaN,Standard Screening - TomoHD,R MLO Breast Tomosynthesis Image,73200000,Extremely dense (>75%),2 - Benign finding,N-Normal interval follow-up,2018-08-16,NaN,NaN,NaN,INDEX-1,15.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,D,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330434...
132,4333002438,56,67560128,1964-07-01,2020-12-12,SCREEN,L,DBT,CC,NaN,STANDARD SCREENING - TOMOHD,L CC Breast Tomosynthesis Image,73200000,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-12-12,NaN,NaN,NaN,INDEX-1,10.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\2.22TB_Tomos_groups3.4....,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4333002...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5935,4337724790,57,455418188,1963-07-01,2021-06-19,SCREEN,R,DBT,MLO,Y,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image Slabs,73500000,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2021-06-19,NaN,NaN,NaN,INDEX-1,12.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4337724...
5936,4337724790,57,455418188,1963-07-01,2021-06-19,SCREEN,R,DBT,CC,NaN,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2021-06-19,NaN,NaN,NaN,INDEX-1,12.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4337724...
5937,4337724790,57,455418188,1963-07-01,2021-06-19,SCREEN,R,DBT,MLO,NaN,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image,73200000,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2021-06-19,NaN,NaN,NaN,INDEX-1,12.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4337724...
5938,4337724790,57,455418188,1963-07-01,2021-06-19,SCREEN,R,DBT,CC,Y,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image Slabs,73500000,Heterogeneously dense (51% - 75%),0 - N

In [30]:
df_invalid.drop_duplicates(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].to_excel('P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Missing imaging data\missing_syn2d_@_index_1.xlsx', index=False)

### **Check 2**: PID with DBT, DBT_Slab, and both

In [31]:
patient_series_df = patient_series.reset_index()
patient_series_df.columns = ['PATIENT_STUDY_ID', 'series_set']
patient_series_df['has_slab'] = patient_series_df['PATIENT_STUDY_ID'].map(patient_has_slab).fillna(False)
patient_series_df['group'] = patient_series_df.apply(categorize, axis=1)

# Merge dates for summary
dates = idx.groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE'].agg(['min', 'max']).reset_index()
dates.columns = ['PATIENT_STUDY_ID', 'earliest', 'latest']
patient_series_df = patient_series_df.merge(dates, on='PATIENT_STUDY_ID', how='left')

print(f"\n{'─'*60}")
print(f"INDEX-1 Series Distribution  (n={len(patient_series_df)} unique patients)")
print(f"{'─'*60}")
for grp_name, grp in patient_series_df.groupby('group'):
    print(f"\n  {grp_name}: {len(grp)} patients")
    print(f"    Earliest exam : {grp['earliest'].min().date()}")
    print(f"    Latest exam   : {grp['latest'].max().date()}")


────────────────────────────────────────────────────────────
INDEX-1 Series Distribution  (n=264 unique patients)
────────────────────────────────────────────────────────────

  Both DBT and DBT_Slab: 39 patients
    Earliest exam : 2020-02-20
    Latest exam   : 2021-12-14

  DBT: 225 patients
    Earliest exam : 2015-07-17
    Latest exam   : 2021-03-20


In [32]:
patient_series_df

,PATIENT_STUDY_ID,series_set,has_slab,group,earliest,latest
0,4330350999,{DBT},False,DBT,2021-01-18,2021-01-18
1,4330421371,"{C VIEW, DBT}",False,DBT,2018-12-03,2018-12-03
2,4330434822,{DBT},False,DBT,2018-08-16,2018-08-16
3,4330449572,"{C VIEW, DBT}",False,DBT,2019-08-24,2019-08-24
4,4330495728,"{C VIEW, DBT}",False,DBT,2018-09-11,2018-09-11
...,...,...,...,...,...,...
259,4335849525,{DBT},True,Both DBT and DBT_Slab,2021-04-23,2021-04-23
260,4337018091,"{DBT, IN2D}",False,DBT,2020-05-27,2020-05-27
261,4337724790,{DBT},True,Both DBT and DBT_Slab,2021-06-19,2021-06-19
262,4337779562,"{DBT, IN2D}",False,DBT,2019-09-13,2019-09-13


## **INDEX**

### **Check 1**: PID with both DBT + (C VIEW / IN2D)

In [33]:
# ── Per-patient series sets at INDEX-1 ───────────────────────────────────────
idx = df[df['Index'] == 'INDEX'].copy()
idx['EXAM_COMPLETED_DATE'] = pd.to_datetime(idx['EXAM_COMPLETED_DATE'], errors='coerce')
patient_series = idx.groupby('PATIENT_STUDY_ID')['Series'].apply(set)
patient_has_slab = idx[idx['Slab'] == 'Y'].groupby('PATIENT_STUDY_ID').size() > 0

valid_pids   = patient_series[patient_series.apply(is_valid)].index
invalid_pids = patient_series[~patient_series.apply(is_valid)].index

print(f"INDEX patients with both DBT + (C VIEW/IN2D) : {len(valid_pids)}")
print(f"INDEX patients MISSING C VIEW/IN2D           : {len(invalid_pids)}")

INDEX patients with both DBT + (C VIEW/IN2D) : 140
INDEX patients MISSING C VIEW/IN2D           : 124


In [34]:
df_invalid = idx[idx['PATIENT_STUDY_ID'].isin(invalid_pids)]
df_invalid

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,PatientBirthDate,StudyDate,Study,SIDE,Series,View,Slab,StudyDescription,SeriesDescription,SeriesNumber,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,time_to_INDEX_months,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,Ki-67_Percent,Ki-67_Grade,subtype,FolderPath,Density,TargetFolderPath
12,4330421371,48,455406763,1973-07-01,2021-07-09,NaN,L,DBT,CC,NaN,BREAST IMAGING TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2021-07-09,465506.0,2021-07-27,Malignant,INDEX,NaN,II,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330421...
13,4330421371,48,455406763,1973-07-01,2021-07-09,NaN,L,DBT,MLO,NaN,BREAST IMAGING TOMOSYNTHESIS,L MLO Breast Tomosynthesis Image,73200000,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2021-07-09,465506.0,2021-07-27,Malignant,INDEX,NaN,II,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330421...
14,4330421371,48,455406763,1973-07-01,2021-07-09,NaN,L,DBT,ML,NaN,BREAST IMAGING TOMOSYNTHESIS,L ML Breast Tomosynthesis Image,73200000,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2021-07-09,465506.0,2021-07-27,Malignant,INDEX,NaN,II,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330421...
15,4330421371,48,455406763,1973-07-01,2021-07-09,NaN,L,DBT,CC,Y,BREAST IMAGING TOMOSYNTHESIS,L CC Breast Tomosynthesis Image Slabs,73500000,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2021-07-09,465506.0,2021-07-27,Malignant,INDEX,NaN,II,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330421...
16,4330421371,48,455406763,1973-07-01,2021-07-09,NaN,L,DBT,MLO,Y,BREAST IMAGING TOMOSYNTHESIS,L MLO Breast Tomosynthesis Image Slabs,73500000,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2021-07-09,465506.0,2021-07-27,Malignant,INDEX,NaN,II,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.21TB_Tomos_Round_12\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4330421...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5880,4334471920,78,454231521,1943-07-01,2021-10-06,SCREEN,R,DBT,CC,Y,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image Slabs,73500000,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2021-10-06,NaN,NaN,NaN,INDEX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.28TB_Tomos_Round_11\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4334471...
5881,4334471920,78,454231521,1943-07-01,2021-10-06,SCREEN,R,DBT,MLO,Y,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image Slabs,73500000,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2021-10-06,NaN,NaN,NaN,INDEX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,J:\Testing\jlee\MO_DBT\1.28TB_Tomos_Round_11\P...,C,J:/Testing/jlee/MO-DBT-Eligible\Cancer\4334471...
5882,4334471920,78,454008009,1943-07-01,2021-10-20,DIAG,L,DBT,CC,Y,DIAGNOSTIC LEFT UNI MAMMO WITH TOMO,L CC Breast Tomosynthesis Image Slabs,73500000,Heterogeneously dense (51% - 75%),4C - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-10-20,463952.0,2021-10-25,Malignant,INDEX,NaN,ID

In [35]:
df_invalid.drop_duplicates(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].to_excel('P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Missing imaging data\missing_syn2d_@_index.xlsx', index=False)

### **Check 2**: PID with DBT, DBT_Slab, and both

In [36]:
patient_series_df = patient_series.reset_index()
patient_series_df.columns = ['PATIENT_STUDY_ID', 'series_set']
patient_series_df['has_slab'] = patient_series_df['PATIENT_STUDY_ID'].map(patient_has_slab).fillna(False)
patient_series_df['group'] = patient_series_df.apply(categorize, axis=1)

# Merge dates for summary
dates = idx.groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE'].agg(['min', 'max']).reset_index()
dates.columns = ['PATIENT_STUDY_ID', 'earliest', 'latest']
patient_series_df = patient_series_df.merge(dates, on='PATIENT_STUDY_ID', how='left')

print(f"\n{'─'*60}")
print(f"INDEX Series Distribution  (n={len(patient_series_df)} unique patients)")
print(f"{'─'*60}")
for grp_name, grp in patient_series_df.groupby('group'):
    print(f"\n  {grp_name}: {len(grp)} patients")
    print(f"    Earliest exam : {grp['earliest'].min().date()}")
    print(f"    Latest exam   : {grp['latest'].max().date()}")


────────────────────────────────────────────────────────────
INDEX Series Distribution  (n=264 unique patients)
────────────────────────────────────────────────────────────

  Both DBT and DBT_Slab: 106 patients
    Earliest exam : 2020-02-18
    Latest exam   : 2022-12-21

  DBT: 139 patients
    Earliest exam : 2016-12-20
    Latest exam   : 2021-04-15

  Neither: 19 patients
    Earliest exam : 2018-02-08
    Latest exam   : 2022-09-14


In [37]:
patient_series_df[patient_series_df["group"]=="Neither"]

,PATIENT_STUDY_ID,series_set,has_slab,group,earliest,latest
63,4333205138,{IN2D},False,Neither,2022-09-14,2022-09-14
65,4333213108,{FFDM},False,Neither,2018-03-06,2018-03-06
95,4333499603,"{FFDM, C VIEW}",False,Neither,2018-04-02,2018-04-02
120,4333690981,{IN2D},False,Neither,2020-05-07,2020-05-07
121,4333693207,{IN2D},False,Neither,2022-05-06,2022-05-06
123,4333728096,"{FFDM, IN2D}",False,Neither,2020-06-19,2020-06-19
128,4333777069,"{FFDM, C VIEW}",False,Neither,2019-04-05,2019-04-05
148,4333967125,{IN2D},False,Neither,2022-05-02,2022-05-02
149,4333977496,"{FFDM, C VIEW}",False,Neither,2019-01-18,2019-01-18
150,4333981936,{IN2D},False,Neither,2020-03-18,2020-03-18


## **overview**

In [38]:
for index_val in ['INDEX', 'INDEX-1']:
    sub = df[df['Index'] == index_val].copy()
    sub['Series'] = sub['Series'].str.strip()
    sub['EXAM_COMPLETED_DATE'] = pd.to_datetime(sub['EXAM_COMPLETED_DATE'], errors='coerce')

    patient_series   = sub.groupby('PATIENT_STUDY_ID')['Series'].apply(set)
    patient_has_slab = sub[sub['Slab'] == 'Y'].groupby('PATIENT_STUDY_ID').size() > 0

    has_dbt_mask = patient_series.apply(lambda s: 'DBT' in s)
    has_2d_mask  = patient_series.apply(lambda s: bool(s & {'C VIEW', 'IN2D'}))

    n_total       = len(patient_series)
    n_has_dbt     = has_dbt_mask.sum()
    n_has_2d      = has_2d_mask.sum()
    n_missing_dbt = (~has_dbt_mask).sum()
    n_valid       = (has_dbt_mask & has_2d_mask).sum()
    n_dbt_no_2d   = (has_dbt_mask & ~has_2d_mask).sum()

    print(f"\n{'='*60}")
    print(f"  {index_val}  —  {n_total} unique patients")
    print(f"{'='*60}")
    print(f"  Has DBT                      : {n_has_dbt} ({n_has_dbt/n_total*100:.1f}%)")
    print(f"  Has C VIEW or IN2D         : {n_has_2d} ({n_has_2d/n_total*100:.1f}%)")
    print(f"  Missing DBT ⚠                : {n_missing_dbt} ({n_missing_dbt/n_total*100:.1f}%)")
    print(f"  Has DBT + (C VIEW or IN2D)   : {n_valid} ({n_valid/n_total*100:.1f}%)")
    print(f"  Has DBT but no C VIEW/IN2D   : {n_dbt_no_2d} ({n_dbt_no_2d/n_total*100:.1f}%)")

    # ── Save flagged patients ─────────────────────────────────────────────────
    id_acc = sub.drop_duplicates('PATIENT_STUDY_ID')[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']]

    if index_val == 'INDEX-1':
        # DBT but no C VIEW/IN2D
        pids_dbt_no2d = patient_series[has_dbt_mask & ~has_2d_mask].index
        idx1_dbt_no2d = id_acc[id_acc['PATIENT_STUDY_ID'].isin(pids_dbt_no2d)].reset_index(drop=True)
        print(f"\n  → Saved: idx1_dbt_no2d  ({len(idx1_dbt_no2d)} patients)")

    elif index_val == 'INDEX':
        # DBT but no C VIEW/IN2D
        pids_dbt_no2d = patient_series[has_dbt_mask & ~has_2d_mask].index
        idx_dbt_no2d  = id_acc[id_acc['PATIENT_STUDY_ID'].isin(pids_dbt_no2d)].reset_index(drop=True)
        print(f"\n  → Saved: idx_dbt_no2d   ({len(idx_dbt_no2d)} patients)")

        # No DBT at all
        pids_no_dbt  = patient_series[~has_dbt_mask].index
        idx_no_dbt   = id_acc[id_acc['PATIENT_STUDY_ID'].isin(pids_no_dbt)].reset_index(drop=True)
        print(f"  → Saved: idx_no_dbt     ({len(idx_no_dbt)} patients)")

    # ── DBT / Slab breakdown ──────────────────────────────────────────────────
    ps_df = patient_series.reset_index()
    ps_df.columns = ['PATIENT_STUDY_ID', 'series_set']
    ps_df['has_slab'] = ps_df['PATIENT_STUDY_ID'].map(patient_has_slab).fillna(False)

    def categorize(row):
        has_dbt  = 'DBT' in row['series_set']
        has_slab = row['has_slab']
        if has_dbt and has_slab: return 'Both DBT and DBT_Slab'
        if has_dbt:              return 'DBT only'
        if has_slab:             return 'DBT_Slab only'
        return 'Neither'

    ps_df['group'] = ps_df.apply(categorize, axis=1)
    dates = sub.groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE'].agg(['min','max']).reset_index()
    dates.columns = ['PATIENT_STUDY_ID', 'earliest', 'latest']
    ps_df = ps_df.merge(dates, on='PATIENT_STUDY_ID', how='left')

    print(f"\n  ── DBT / Slab breakdown ──────────────────────────────")
    for grp_name, grp in ps_df.groupby('group'):
        print(f"  {grp_name}: {len(grp)} patients")
        print(f"    Earliest : {grp['earliest'].min().date()}  |  Latest : {grp['latest'].max().date()}")


  INDEX  —  264 unique patients
  Has DBT                      : 245 (92.8%)
  Has C VIEW or IN2D         : 158 (59.8%)
  Missing DBT ⚠                : 19 (7.2%)
  Has DBT + (C VIEW or IN2D)   : 140 (53.0%)
  Has DBT but no C VIEW/IN2D   : 105 (39.8%)

  → Saved: idx_dbt_no2d   (105 patients)
  → Saved: idx_no_dbt     (19 patients)

  ── DBT / Slab breakdown ──────────────────────────────
  Both DBT and DBT_Slab: 106 patients
    Earliest : 2020-02-18  |  Latest : 2022-12-21
  DBT only: 139 patients
    Earliest : 2016-12-20  |  Latest : 2021-04-15
  Neither: 19 patients
    Earliest : 2018-02-08  |  Latest : 2022-09-14

  INDEX-1  —  264 unique patients
  Has DBT                      : 264 (100.0%)
  Has C VIEW or IN2D         : 196 (74.2%)
  Missing DBT ⚠                : 0 (0.0%)
  Has DBT + (C VIEW or IN2D)   : 196 (74.2%)
  Has DBT but no C VIEW/IN2D   : 68 (25.8%)

  → Saved: idx1_dbt_no2d  (68 patients)

  ── DBT / Slab breakdown ──────────────────────────────
  Both DBT and D

In [39]:
idx_dbt_no2d.to_excel('P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Missing imaging data\missing_syn2d_@_index.xlsx', index=False)

In [40]:
idx_no_dbt.to_excel('P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Missing imaging data\missing_dbt_syn2d_@index.xlsx', index=False)

In [41]:
idx1_dbt_no2d.to_excel('P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\Missing imaging data\missing_syn2d_@_index_1.xlsx', index=False)